# Проверка и установка отсутствующих библиотек

In [2]:
import subprocess
import sys
import importlib
import warnings
warnings.filterwarnings('ignore')

required_libs = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'scipy': 'scipy',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'requests': 'requests',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl'
}

missing = []
for lib, pkg in required_libs.items():
    try:
        importlib.import_module(lib)
    except ImportError:
        missing.append(pkg)

if missing:
    print("Установка недостающих библиотек:", missing)
    for pkg in missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
    print("✅ Готово. Перезапустите ячейку, если нужно.")
else:
    print("✅ Все необходимые библиотеки уже установлены.")

✅ Все необходимые библиотеки уже установлены.


# Импорт модулей и создание папок

In [4]:
import os
import time
from datetime import datetime
import pandas as pd
from tqdm import tqdm

from config import PERIODS, MOEX_PAUSE_BETWEEN_TICKERS
from utils import (
    get_all_tickers, get_historical_prices,
    load_prices_dict, save_prices_dict, ensure_dirs
)

# Создаём папку data (и, если нужно, другие)
ensure_dirs(['data'])

# Функция загрузки периода с проверкой на существование файла (файла для быстрого анализа представлен)

In [6]:
def download_period(period_name, period_config, force_reload=False):
    pkl_file = os.path.join('data', period_config['pkl_file'])
    
    # Если файл уже существует и не требуется перезагрузка – пропускаем
    if not force_reload and os.path.exists(pkl_file):
        try:
            test_dict = load_prices_dict(pkl_file)
            if len(test_dict) > 10:
                print(f"✅ Данные для {period_name} уже загружены ({len(test_dict)} тикеров). Пропускаем.")
                return
        except:
            print(f"⚠️ Файл {pkl_file} повреждён, перезагружаем.")
    
    print(f"\nЗагрузка {period_name}: {period_config['start']} – {period_config['end']}")
    all_tickers = get_all_tickers()
    print(f"Всего тикеров: {len(all_tickers)}")
    
    start_date = datetime.strptime(period_config['start'], '%Y-%m-%d')
    end_date = datetime.strptime(period_config['end'], '%Y-%m-%d')
    
    prices_dict = {}
    for tkr in tqdm(all_tickers, desc=period_name):
        prices = get_historical_prices(tkr, start_date, end_date)
        if prices is not None and len(prices) > 0:
            prices_dict[tkr] = prices
        time.sleep(MOEX_PAUSE_BETWEEN_TICKERS)
    
    if not prices_dict:
        raise ValueError(f"Не удалось загрузить ни одной акции для {period_name}.")
    
    save_prices_dict(pkl_file, prices_dict)
    print(f"✅ Сохранён {pkl_file} ({len(prices_dict)} тикеров)")

# Запуск загрузки для всех периодов

In [8]:
for period_name, period_config in PERIODS.items():
    download_period(period_name, period_config, force_reload=False)

print("\n" + "="*60)
print("ЗАГРУЗКА ЗАВЕРШЕНА. ПРОВЕРЬТЕ, ЧТО СЛОВАРИ СОХРАНЕНЫ В ПАПКУ 'data'.")
print("="*60)

✅ Данные для 2023_2024 уже загружены (242 тикеров). Пропускаем.
✅ Данные для 2024_2025 уже загружены (255 тикеров). Пропускаем.

ЗАГРУЗКА ЗАВЕРШЕНА. ПРОВЕРЬТЕ, ЧТО СЛОВАРИ СОХРАНЕНЫ В ПАПКУ 'data'.
